# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets in the dataset by their @id and fields
record_sets = list(metadata.record_sets)
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for f in rs.fields:
            print(f"    {f.name}: @id={f.id} (type: {f.dataType})")
        print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets found
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    if len(records):
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print("No records found.")
    print("")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For EDA, select the first available record set with data, and apply steps
selected_df = None
selected_record_set = None
for rs_id, df in dataframes.items():
    if len(df) > 0:
        selected_df = df.copy()
        selected_record_set = rs_id
        break
if selected_df is None:
    print('No record set with data available for EDA.')
else:
    print(f"Using record set: {selected_record_set}\nColumns: {selected_df.columns.tolist()}")
    # Try to select a numeric field for EDA (heuristically pick 'Abundance', 'MW', 'Coverage', etc.)
    potential_numeric_fields = [c for c in selected_df.columns if selected_df[c].dtype in ['float64', 'int64'] or 'abundance' in c.lower() or 'count' in c.lower() or 'cov' in c.lower() or 'mw' in c.lower()]
    if not potential_numeric_fields:
        print('No obvious numeric field was found.')
    else:
        # Try picking the first such field
        numeric_field = potential_numeric_fields[0]
        print(f"Selected numeric field for EDA: {numeric_field}")

        # Convert to numeric, coerce errors
        selected_df[numeric_field] = pd.to_numeric(selected_df[numeric_field], errors='coerce')
        # Filter out non-numeric rows
        df_nonan = selected_df[selected_df[numeric_field].notna()]
        threshold = df_nonan[numeric_field].median() if len(df_nonan) else 0
        filtered_df = df_nonan[df_nonan[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > median ({threshold}):")
        display(filtered_df.head())
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field
        potential_group_fields = [c for c in selected_df.columns if c != numeric_field and selected_df[c].dtype == object and not c.lower().endswith('id')]
        if potential_group_fields:
            group_field = potential_group_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            display(grouped_df.head())
        else:
            print('No suitable categorical field for grouping found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_df is not None and potential_numeric_fields:
    # Plot histogram of the selected numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(selected_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If there is a grouping field, show boxplot
    if potential_group_fields:
        plt.figure(figsize=(10,5))
        # Limit categories if too many
        count_per_cat = selected_df[potential_group_fields[0]].value_counts().sort_values(ascending=False)
        cats = count_per_cat.index[:10]
        subset = selected_df[selected_df[potential_group_fields[0]].isin(cats)]
        sns.boxplot(data=subset, x=potential_group_fields[0], y=numeric_field)
        plt.title(f'{numeric_field} by {potential_group_fields[0]}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset, describing proteins in extracellular vesicles from human mast cells, is rich in metadata and tabular results.
- Using `mlcroissant`, we were able to load the Croissant package, list its record sets, and extract data directly by referencing their Croissant `@id`.
- Exploratory data analysis demonstrated filtering, normalization, and grouping on numeric protein features such as abundance or molecular weight across potential biological groupings.
- Visualizing the numeric field revealed the data's distribution and possible grouping patterns for further downstream biological interpretation.
- For more advanced analysis, one could integrate external ontologies, link to Uniprot, or perform domain-specific enrichment and clustering.